# E3 · Adaptación métrica para identidades de perros y gatos

E1 estudia la transferencia directa de CLIP y E2 añade localización. E3 investiga si una cabeza MLP, entrenada con individuos públicos, mejora la recuperación de mascotas locales sin actualizar YOLO ni CLIP.

**Hipótesis.** La proyección aprendida mejora la recuperación de identidades privadas frente a E1/E2. Cada métrica y especie se contrasta por separado; ni menor pérdida ni mejor validación pública demuestran transferencia por sí solas.

| Decisión | Protocolo E3 |
|---|---|
| Representación | E2 → MLP 512→256→512 → normalización L2 |
| Fuentes propuestas | Dogs of the World (perros) y LCW (gatos), versiones 1 |
| Unidad de etiqueta | Individuo, nunca raza; JSON de identidad en perros y carpeta de individuo en gatos |
| Selección de fotos | Piloto acotado, filtro YOLO, exclusiones y mínimo de 4 fotos distintas por individuo |
| Optimización | Batch-Hard Triplet Loss, margen 0.2; lotes P×K de una especie, alternando perros y gatos |
| Checkpoint | mAP macro por especie en validación pública, identidades disjuntas de train |
| Comparación local | E1/E2/E3 sobre el mismo commit publicado de PetTrace v4 |
| Publicación | `LOG_TO_MLFLOW = False`; registro de modelo separado y desactivado |
| Estado | Implementación disponible; entrenamiento real pendiente |

El contraste v4 es un benchmark exploratorio ya observado en E1/E2. No se presenta como prueba final nunca consultada. Los resultados se generan con la ejecución; no hay métricas reales de E3 todavía.

La muestra pública procede del pipeline de datos externo; esta corrida consume su versión y registra la huella utilizada.


## Entorno y configuración reproducible

Las versiones, fuentes del paquete y controles quedan visibles. PyTorch optimiza la cabeza en GPU; ONNX Runtime CPU mantiene el mismo camino de inferencia que E1/E2. Una ejecución remota necesita el paquete actualizado publicado o instalado mediante una wheel; los cambios locales del repositorio no llegan automáticamente a Colab.

La publicación y el registro están desactivados mediante valores literales, sin activación implícita desde variables del entorno. MLflow sigue siendo necesario en modo lectura para descargar los modelos existentes. Las credenciales permanecen fuera del notebook.

Colab instala el paquete actualizado desde GitHub. `PETTRACE_PACKAGE_SOURCE` permite seleccionar explícitamente una wheel o una fuente alternativa. El runtime debe estar limpio después de reemplazar paquetes ya importados.


In [ ]:
# E3 runtime dependencies.
%pip install -q --only-binary=:all: \
    "numpy>=2.1,<2.3" \
    "pillow>=11,<13" \
    "pyyaml>=6,<7" \
    "onnxruntime==1.20.1" \
    "mlflow-skinny==3.16.1" \
    "huggingface-hub>=1.32,<2" \
    "matplotlib>=3.9,<4" \
    "torch>=2.5,<3" \
    "onnx>=1.17,<2" \
    "pandas>=2.2.3,<4" \
    "ipywidgets>=8.1,<9"


In [ ]:
import importlib.util
import os
import torch  # Import before the ONNX modules used by the prepared-data loader.

PACKAGE_SOURCE = os.environ.get("PETTRACE_PACKAGE_SOURCE", "git+https://github.com/alessandro54/pettrace-ml.git@main")
try:
    has_preparation = all(importlib.util.find_spec(name) is not None
                          for name in ("pettrace.e3_prepare", "pettrace.training_data"))
except ModuleNotFoundError:
    has_preparation = False
if not has_preparation or os.environ.get("PETTRACE_PACKAGE_SOURCE"):
    get_ipython().run_line_magic("pip", f'install -q --upgrade --force-reinstall --no-deps "{PACKAGE_SOURCE}"')
importlib.invalidate_caches()
if not all(importlib.util.find_spec(name) is not None
           for name in ("pettrace.e3_prepare", "pettrace.training_data")):
    raise RuntimeError("Install the current pettrace-ml package and restart the kernel.")
import pettrace
print("pettrace-ml:", pettrace.__version__)


In [ ]:
# Request access only after verifying the prepared dataset; never print credential values.
def configure_private_access(names=("HF_TOKEN", "MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD")):
    # Credentials stay in environment variables / Secrets / hidden prompts, never in notebook source.
    import getpass
    import os
    import warnings

    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    for name in names:
        if not os.environ.get(name) and userdata is not None:
            try:
                os.environ[name] = userdata.get(name)
            except Exception:
                pass  # VS Code may not expose Secrets; use a hidden prompt below.
        if not os.environ.get(name):
            if name == "MLFLOW_TRACKING_URI":
                value = "https://ml.pettrace.app"
            else:
                # Fail before reading input if the kernel cannot hide the entered value.
                with warnings.catch_warnings():
                    warnings.simplefilter("error", getpass.GetPassWarning)
                    value = getpass.getpass(f"{name}: ")
            if not value.strip():
                raise ValueError(f"Missing {name}.")
            os.environ[name] = value


In [ ]:
import json
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import mlflow
import numpy as np
import torch
import onnxruntime as ort
from IPython.display import Markdown, display
from huggingface_hub import HfApi, snapshot_download

from pettrace import data
from pettrace.e3 import Config, fit, export_encoder
from pettrace.e3_prepare import load_prepared
from pettrace.evaluation import evaluate, load_model
from pettrace.figures import save
from pettrace.public_data import feature_cache, sha256
from pettrace.tracking import PRODUCTION_URI, publish_evaluation, save_provenance, source_identity

EXPERIMENT = "E3"
DATASET = "v4"
DATASET_COMMIT = "8b12db0dce41040a821f9792bea5b352342263ce"
BASE_MODEL = "models:/pettrace-embedder/2"
BASELINE_MODEL = "models:/pettrace-embedder/1"
ENCODER_SHA256 = "4773e649274ca8774c4a63d45f0438fe598ee7ba67e65def97537aa0eca9a576"
DETECTOR_SHA256 = "da8f9b9ebea5157cdcefa7bea9fd782249ebdf119f917c07d64e5b41942b9c9d"
LOG_TO_MLFLOW = False
REGISTER_E3 = False
SEED = 42
DATA_ROOT = Path(os.environ.get("E3_DATA_ROOT", "datasets/public/e3-body"))
shared_version = Path("datasets/build/pettrace-training/v1.json")
PREPARED_DATASET_ROOT = Path(os.environ.get("E3_PREPARED_DATASET", str(
    shared_version if shared_version.exists() else DATA_ROOT / "prepared/latest.json")))
TRAINING_DATASET_REPO = os.environ.get("HF_TRAINING_DATASET_REPO", "")
TRAINING_DATASET_VERSION = os.environ.get("HF_TRAINING_DATASET_VERSION", "v1")
TRAINING_DATASET_REVISION = os.environ.get("HF_TRAINING_DATASET_REVISION", TRAINING_DATASET_VERSION)
TRAINING_DATASET_FINGERPRINT = os.environ.get("HF_TRAINING_DATASET_FINGERPRINT")
FEATURE_CACHE_ROOT = Path(os.environ.get("E3_FEATURE_CACHE", str(DATA_ROOT / "feature-cache")))
RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
REPORT = Path(os.environ.get("PETTRACE_OUTPUT_DIR", "reports")) / f"e3_{DATASET}" / RUN_LABEL
REPORT.mkdir(parents=True, exist_ok=False)
SOURCE_IDENTITY = source_identity()
SOURCE_SHA256 = SOURCE_IDENTITY["source_sha256"]
mlflow.set_tracking_uri(PRODUCTION_URI)
mlflow.set_registry_uri(PRODUCTION_URI)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Package source:", SOURCE_IDENTITY)
print("Training device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Prepared dataset:", PREPARED_DATASET_ROOT)
print("Report:", REPORT)
print("Publish to MLflow:", LOG_TO_MLFLOW, "| Register model:", REGISTER_E3)


## 1. Dataset de entrenamiento versionado

La muestra procede de `pettrace-training`, preparado de forma independiente en `datasets/prepare_training.ipynb`. Puede reutilizarse en otros experimentos. La versión conserva fotos, splits por individuo, reglas de limpieza, exclusiones y hashes; el experimento verifica su integridad antes de entrenar.

Localmente se carga `datasets/build/pettrace-training/v1.json` o una ruta fijada con `E3_PREPARED_DATASET`. Para una versión publicada se configuran `HF_TRAINING_DATASET_REPO`, `HF_TRAINING_DATASET_VERSION` y `HF_TRAINING_DATASET_REVISION` (preferiblemente el commit de la publicación); `HF_TRAINING_DATASET_FINGERPRINT` verifica la muestra esperada. La referencia se resuelve una vez y la descarga se fija al commit registrado en el reporte.

`pettrace-reid@v4` mantiene su función de benchmark privado; no aporta fotos al dataset de entrenamiento.


In [ ]:
from pettrace.training_data import load_local_training, load_training

if TRAINING_DATASET_REPO:
    configure_private_access(names=("HF_TOKEN",))
    prepared, training_source = load_training(
        repo_id=TRAINING_DATASET_REPO, version=TRAINING_DATASET_VERSION, revision=TRAINING_DATASET_REVISION,
        expected_fingerprint=TRAINING_DATASET_FINGERPRINT)
else:
    prepared, training_source = load_local_training(PREPARED_DATASET_ROOT)
manifest_rows = prepared.rows
manifest_path = prepared.root / "manifest.csv"
review_path = prepared.root / "review.csv"
audit_summary = json.loads((prepared.root / "audit.json").read_text())
detection_audit = json.loads((prepared.root / "animal-screening.json").read_text())
for filename in ("manifest.csv", "review.csv", "audit.json", "sources.json", "animal-screening.json", "prepared.json"):
    shutil.copy2(prepared.root / filename, REPORT / filename)
(REPORT / "data-preparation.json").write_text(json.dumps(
    {"prepared_dataset_fingerprint": prepared.fingerprint, "training_source": training_source,
     "root": str(prepared.root),
     "prepared_receipt_sha256": sha256(prepared.root / "prepared.json"),
     "manifest_sha256": sha256(manifest_path), "counts": prepared.summary["counts"]}, indent=2))
print("Prepared dataset fingerprint:", prepared.fingerprint)
print("Photos:", len(manifest_rows), "| Identities:", prepared.summary["retained_identities"])
print(json.dumps(prepared.summary["counts"], indent=2))
print(prepared.summary["limitation"])


### 1.1 Benchmark local fijado a v4

La descarga privada se fija al mismo commit de E1/E2 v4. Ninguna mascota privada participa de la preparación pública, el entrenamiento o la selección del checkpoint. Se comprueba el solapamiento exacto de imágenes con el dataset preparado.


In [ ]:
configure_private_access()
resolved_commit = HfApi().dataset_info(data.REPO, revision=DATASET).sha
assert resolved_commit == DATASET_COMMIT, "The v4 tag changed; review the comparison protocol"
private_root = snapshot_download(data.REPO, repo_type="dataset", revision=DATASET_COMMIT)
ds = data.from_dir(private_root, repo=data.REPO, version=DATASET, commit=DATASET_COMMIT)
private_hashes = {sha256(photo.path) for photo in ds.photos}
assert not private_hashes.intersection(row["sha256"] for row in manifest_rows), "Public/private image overlap"
print("Benchmark:", ds.name, "| Commit:", ds.commit, "| Photos:", len(ds.photos))
for species in ("cat", "dog"):
    photos = [photo for photo in ds.photos if photo.species == species]
    print(species, "identities:", len({photo.animal for photo in photos}), "photos:", len(photos))


## 2. Extractor E2 congelado

El entrenamiento reutiliza E2 sin actualizar detector ni encoder. Sus hashes y golden set se verifican de nuevo, independientemente del detector empleado en la preparación. Solo se ajusta la cabeza MLP.


In [ ]:
base_model = load_model(BASE_MODEL)
assert base_model.golden_ok, "E2 golden validation failed"
assert base_model.sha256 == {"encoder.onnx": ENCODER_SHA256, "detector.onnx": DETECTOR_SHA256}
assert base_model.pipeline.detector is not None, "E3 requires the E2 crop pipeline"
print("Base:", base_model.uri, "| Golden:", base_model.golden_fingerprint)


## 3. Diseño del aprendizaje métrico

La cabeza MLP transforma vectores de 512 dimensiones mediante una capa intermedia de 256, ReLU y normalización L2 a la salida. La pérdida acerca ejemplos de la misma identidad y separa identidades distintas:

$$\mathcal{L}=\frac{1}{|A|}\sum_{a\in A}\max\left(0,\max_{p\ne a,\ y_p=y_a}d(z_a,z_p)-\min_{n:\ y_n\ne y_a}d(z_a,z_n)+m\right),\quad m=0.2.$$

El muestreo P×K reúne 8 identidades y 4 imágenes por identidad, alternando lotes de gatos y perros. Los positivos y negativos se seleccionan dentro del lote y de la misma especie. La existencia de distintas fotos no certifica que procedan de sesiones diferentes.

La configuración implementada utiliza AdamW, tasa de aprendizaje 1e-3, weight decay 1e-4, hasta 30 épocas y paciencia 7. Los embeddings de E2 se cachean porque el extractor no cambia. El checkpoint se selecciona exclusivamente con mAP macro por especie de la validación pública; la muestra privada no selecciona parámetros ni época.

La exportación fusiona encoder y cabeza en ONNX y verifica el artefacto resultante. Estas verificaciones controlan la implementación, sin sustituir una corrida real ni demostrar mejora de recuperación.

La inicialización y el muestreo usan semilla 42. Antes de entrenar, el reporte conserva la configuración, el manifiesto revisado, el hash de caché y la identidad exacta del extractor. Las curvas de pérdida y validación se muestran después del ajuste.


In [ ]:
config = Config(seed=SEED, epochs=30, steps_per_epoch=100, identities_per_batch=8,
                images_per_identity=4, hidden=256, margin=0.2, learning_rate=1e-3,
                weight_decay=1e-4, patience=7)
assert device == "cuda", "Select a GPU runtime for the real E3 training run"
training_rows = [row for row in manifest_rows if row["split"] in {"train", "val"}]
features, cache_digest = feature_cache(base_model.pipeline, training_rows, FEATURE_CACHE_ROOT, base_model.dir)
from dataclasses import asdict
run_config = {"experiment": EXPERIMENT, "seed": SEED, "training": asdict(config),
              "prepared_dataset_fingerprint": prepared.fingerprint, "training_dataset_source": training_source,
              "dataset": ds.name, "dataset_commit": ds.commit, "dataset_fingerprint": ds.fingerprint,
              "base_model": base_model.uri, "base_model_sha256": base_model.sha256,
              "cache_sha256": cache_digest, "manifest_sha256": sha256(REPORT / "manifest.csv"),
              "review_sha256": sha256(REPORT / "review.csv"),
              "animal_screening_sha256": sha256(REPORT / "animal-screening.json"), "source": SOURCE_IDENTITY,
              "log_to_mlflow": LOG_TO_MLFLOW, "register_model": REGISTER_E3}
(REPORT / "training-config.json").write_text(json.dumps(run_config, indent=2))
trained_head, train_summary = fit(features, training_rows, REPORT, config=config, device=device)
(REPORT / "training-summary.json").write_text(json.dumps(train_summary, indent=2))
print("Selected epoch:", train_summary["best_epoch"], "| Public validation macro mAP:", train_summary["best_val_mAP"])


In [ ]:
import matplotlib.pyplot as plt
history = train_summary["history"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot([row["epoch"] for row in history], [row["train.loss"] for row in history])
axes[0].set(xlabel="Época", ylabel="Triplet loss", title="Optimización pública")
axes[1].plot([row["epoch"] for row in history], [row["val.macro_species_mAP"] for row in history])
axes[1].axvline(train_summary["best_epoch"], color="gray", linestyle="--")
axes[1].set(xlabel="Época", ylabel="mAP macro", title="Selección en validación pública")
fig.tight_layout()
fig.savefig(REPORT / "training-curves.png", dpi=160)
plt.show()


In [ ]:
model_dir = REPORT / "model"
onnx_hashes = export_encoder(base_model.dir, trained_head, model_dir, model_version="clip-vit-b32-crop-mlp@v1")
print("Exported E3 artifacts:", onnx_hashes)


## 4. Protocolo de evaluación y alcance de la comparación

El artefacto fusionado E3 se evalúa sobre `pettrace-reid@v4` mediante el evaluador compartido y una reconstrucción visible del ranking. Se excluye la propia foto de la galería, se restringen candidatos a la misma especie y se utiliza el máximo score por identidad para Rank-k. AP/mAP conserva el ranking por fotografías.

La evaluación usa todas las fotos elegibles de v4. E1 y E2 se recalculan dentro del notebook sobre esa misma muestra. Las líneas base se recalculan sobre el mismo objeto dataset y sus consultas se comparan explícitamente.

La EER mostrada se ajusta sobre los mismos pares privados en los que se reporta, por lo que es descriptiva. No representa calibración independiente ni un umbral validado para despliegue. El notebook tampoco presenta métricas del test público reservado; el runner extendido tiene un alcance distinto documentado en el protocolo de E3.


In [ ]:
e3_model = load_model(str(model_dir))
assert e3_model.golden_ok, "El modelo E3 fusionado no superó la verificación golden."
ev = evaluate(ds, e3_model.pipeline, zero_shot=True)
E = ev.embeddings
print(f"Embeddings calculados: {E.shape}, norma media = {np.linalg.norm(E, axis=1).mean():.4f}")


In [ ]:
animals = np.array([p.animal for p in ds.photos])
species = np.array([p.species for p in ds.photos])
S = E @ E.T  # Similitud coseno entre todas las fotos

# Evaluación leave-one-out dentro de la misma especie
rows_eval_calc = []
for q in range(len(E)):
    gallery = np.where((species == species[q]) & (np.arange(len(E)) != q))[0]
    if len(set(animals[gallery])) < 2 or not (animals[gallery] == animals[q]).any():
        continue
    sub = S[q, gallery]
    sub_animals = animals[gallery]
    sub_species = species[gallery]

    unique_ids = []
    seen = set()
    for a in sub_animals:
        if a not in seen:
            seen.add(a)
            unique_ids.append(a)
    scores = np.array([sub[sub_animals == a].max() for a in unique_ids])
    order = np.argsort(-scores, kind="stable")
    ordered_ids = np.array(unique_ids)[order]
    rank = int(np.where(ordered_ids == animals[q])[0][0]) + 1

    # Average Precision
    photo_order = np.argsort(-sub, kind="stable")
    labels = (sub_animals[photo_order] == animals[q]).astype(int)
    k = np.arange(1, len(labels) + 1)
    prec = np.cumsum(labels) / k
    ap = float((prec * labels).sum() / max(1, labels.sum()))

    rows_eval_calc.append({"query": q, "animal": animals[q], "species": species[q], "rank": rank, "ap": ap})

ranks = np.array([r["rank"] for r in rows_eval_calc])
aps = np.array([r["ap"] for r in rows_eval_calc])

print("Resultados calculados a mano:")
print(f"  Rank-1: {np.mean(ranks <= 1)*100:.1f} %")
print(f"  Rank-5: {np.mean(ranks <= 5)*100:.1f} %")
print(f"  mAP   : {aps.mean():.3f}")


In [ ]:
expected_ranks = {row["query"]: row["rank"] for row in ev.result["test"]["per_query"]}
assert all(row["rank"] == expected_ranks[row["query"]] for row in rows_eval_calc)
assert np.isclose(aps.mean(), ev.result["test"]["global"]["mAP"])
print("Verificación del cálculo visible con pettrace.evaluation: OK")


## 5. Resultados y comparación controlada

E1 y E2 se recalculan sobre el mismo snapshot y se verifica la coincidencia de consultas y galerías. La tabla presenta globales y resultados por especie; la figura de diferencias separa E3−E1 y E3−E2. Rank-1/5 se expresan en puntos porcentuales y mAP en diferencia absoluta.

La EER permanece descriptiva, estimada sobre los mismos pares privados. Las curvas públicas describen aprendizaje y selección; las métricas locales describen transferencia. No se reporta significancia ni generalización entre sesiones a partir de estas comparaciones.


In [ ]:
s3 = save(ev, e3_model, str(REPORT))
baseline_summaries = {}
for label, uri in {"E1": BASELINE_MODEL, "E2": BASE_MODEL}.items():
    baseline = base_model if label == "E2" else load_model(uri)
    assert baseline.golden_ok and baseline.sha256["encoder.onnx"] == ENCODER_SHA256
    if label == "E1":
        assert baseline.pipeline.detector is None
    baseline_ev = evaluate(ds, baseline.pipeline, zero_shot=True)
    assert [(r["query"], r["animal"], r["gallery_animals"]) for r in baseline_ev.result["test"]["per_query"]] == [
        (r["query"], r["animal"], r["gallery_animals"]) for r in ev.result["test"]["per_query"]]
    baseline_summaries[label] = save(baseline_ev, baseline, str(REPORT / f"baseline_{label.lower()}"))
summaries = {**baseline_summaries, "E3": s3}
lines = ["| Model | Scope | Queries | Rank-1 | Rank-5 | mAP |", "|---|---|---:|---:|---:|---:|"]
for label, summary in summaries.items():
    for scope in ("global", "cat", "dog"):
        values = summary["global"] if scope == "global" else summary["per_species"][scope]
        lines.append(f"| {label} | {scope} | {values['queries']} | {values['rank1']:.2%} | {values['rank5']:.2%} | {values['mAP']:.4f} |")
display(Markdown("\n".join(lines)))
eer_lines = ["| Model | Descriptive EER | Threshold fitted on these pairs |", "|---|---:|---:|"]
for label, summary in summaries.items():
    eer_lines.append(f"| {label} | {summary['eer']:.2%} | {summary['threshold']:.4f} |")
display(Markdown("\n".join(eer_lines)))
comparison = {label: {key: s3["global"][key] - summary["global"][key]
                     for key in ("rank1", "rank5", "mAP")} for label, summary in baseline_summaries.items()}
(REPORT / "comparison.json").write_text(json.dumps(comparison, indent=2))


In [ ]:
captions = {
    "confusion": "Filas: identidad real; columnas: primera identidad recuperada. Las confusiones localizan casos, sin explicar su causa.",
    "rank1_por_animal": "Rank-1 por identidad: muestra cómo se distribuyen los aciertos entre mascotas.",
    "distribucion_scores": "Scores positivos y negativos. La EER y su umbral se estiman sobre estos mismos pares; no son calibración independiente.",
    "cmc": "CMC: fracción de consultas cuya identidad correcta aparece entre los primeros k candidatos.",
    "matriz_similitud": "Similitud entre embeddings; las agrupaciones deben interpretarse junto a etiquetas y confusiones.",
}
for name, caption in captions.items():
    image_path = REPORT / f"{name}.png"
    if image_path.exists():
        display(Image(str(image_path), width=760))
        display(Markdown(caption))
findings = ["# E3 · Hallazgos de la ejecución", "",
            f"Dataset: {ds.name}; commit: {ds.commit}.",
            f"Checkpoint: época {train_summary['best_epoch']}, mAP macro público {train_summary['best_val_mAP']:.4f}.", ""]
for label, delta in comparison.items():
    findings.append(f"- E3 − {label}: Rank-1 {delta['rank1']*100:+.2f} pp; Rank-5 {delta['rank5']*100:+.2f} pp; mAP {delta['mAP']:+.4f}.")
findings.extend(["", "Contraste descriptivo; no establece significancia, efecto causal del fondo, ni independencia entre sesiones."])
findings_text = "\n".join(findings)
(REPORT / "findings.md").write_text(findings_text)
display(Markdown(findings_text))


## 6. Trazabilidad y publicación opcional

Los artefactos se conservan localmente bajo un directorio nuevo por corrida: configuración, procedencia de fuentes, revisión, manifiesto, auditoría, historia/checkpoint, curvas, modelo ONNX, líneas base y resultados. El recibo `publication.json` indica si el resultado se publicó.

`LOG_TO_MLFLOW = False` mantiene la corrida local. `REGISTER_E3 = False` evita registrar una versión; requiere activar explícitamente ambas opciones para hacerlo. La descarga de modelos usa MLflow en modo lectura. La publicación, cuando se habilita, utiliza el experimento existente sin promover un alias.

Las hojas de contacto permanecen en el piloto local. No se copian al reporte ni se publican fotos; el manifiesto de evidencia conserva identificadores/rutas y hashes, no los campos personales de las fuentes.


In [ ]:
if REGISTER_E3 and not LOG_TO_MLFLOW:
    raise ValueError("Model registration requires explicit LOG_TO_MLFLOW = True")
save_provenance(REPORT, e3_model)
receipt = REPORT / "publication.json"
receipt.write_text(json.dumps({"published": False, "run_id": None}))
if LOG_TO_MLFLOW:
    run_id = publish_evaluation(ev, e3_model, REPORT, SOURCE_SHA256, experiment=EXPERIMENT,
                                expected_dataset_version=DATASET, expected_protocol="zero-shot")
    receipt.write_text(json.dumps({"published": True, "run_id": run_id}))
    print("MLflow run:", run_id)
    if REGISTER_E3:
        version = mlflow.register_model(f"runs:/{run_id}/report/model", "pettrace-embedder")
        client = mlflow.MlflowClient()
        client.set_model_version_tag("pettrace-embedder", version.version, "experiment", "E3")
        client.set_model_version_tag("pettrace-embedder", version.version, "status", "candidate")
        print("Registered E3 version:", version.version)
else:
    print("MLflow publication disabled. Local evidence:", REPORT)


## 7. Discusión pendiente y siguientes preguntas

E3 constituye una hipótesis implementada, aún sin resultados de entrenamiento real. La discusión deberá responder si la adaptación transfiere a identidades privadas y qué animales concentran ganancias o pérdidas frente a las líneas base recalculadas.

La interpretación requiere distinguir tres niveles de evidencia: optimización de la cabeza, selección en validación pública y recuperación en PetTrace. Ninguno sustituye a los demás. La similitud entre tomas de una misma sesión limita la fuerza de una conclusión sobre reencuentros en otra ocasión.

La investigación posterior deberá evaluar diversidad de captura, posibles atajos de contexto, confusiones entre individuos parecidos e incertidumbre por identidad. La decisión de desplegar requerirá además un criterio de aceptación y una calibración independientes; este notebook no los establece.
